## Setup

In [1]:
from google.colab import userdata
access_token = userdata.get('CASM-NER')

In [2]:
%%capture
!pip install transformers
!pip install sentencepiece
!pip install seqeval
!pip install datasets
# !pip install git+https://github.com/ay94/multilingual-ner.git

In [ ]:
# from ner import evaluation

In [3]:
## Mount GDrive
from google.colab import drive
drive.mount('/content/drive/', force_remount=True)

## Imports
import os
import sys
import nltk
import time
import torch
import random
import subprocess
import numpy as np
import pandas as pd
import datetime as dt
from itertools import groupby
from tqdm.notebook import tqdm
from datasets import load_dataset
from transformers import pipeline
from collections import Counter, defaultdict
from torch.utils.data import DataLoader, Dataset
from transformers import AutoModelForTokenClassification, AutoTokenizer
from seqeval.metrics import f1_score as seq_f1, precision_score as seq_precision, recall_score as seq_recall, classification_report as seq_classification
from sklearn.metrics import f1_score as skl_f1, precision_score as skl_precision, recall_score as skl_recall, classification_report as skl_classification

Mounted at /content/drive/


In [4]:
# Append the library files into the notebook system path for import
sys.path.append('/content/drive/Shareddrives/Machine Translation/Model benchmarking/Libraries/1.0.2')
# import custom library files
import ner, utils

## Load datasets

### wikiann

In [5]:
# from ner.dataset_base import HuggingFaceMultilingualDataset
# class Wikiann(HuggingFaceMultilingualDataset):
#     dataset_name = 'wikiann'
#     language = 'ro'
#     license = 'unknown'

# dataset = Wikiann()
# dataset.check_labels()

In [6]:
wikiann_label_map = {
    "O": 0,
    "B-PER": 1,
    "I-PER": 2,
    "B-ORG": 3,
    "I-ORG": 4,
    "B-LOC": 5,
    "I-LOC": 6
}

wikiann = ner.ReadNERData()
wikiann_words, wikiann_labels = wikiann.read_dataset('wikiann', wikiann_label_map, lang='ro')

/usr/local/lib/python3.10/dist-packages/huggingface_hub/utils/_token.py:72: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


Generating validation split:   0%|          | 0/10000 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/10000 [00:00<?, ? examples/s]

Generating train split:   0%|          | 0/20000 [00:00<?, ? examples/s]

Generating test Split


  0%|          | 0/10000 [00:00<?, ?it/s]

In [7]:
print(ner.check_labels(wikiann_labels))
# Dataset Label Map Alignment to LOC, ORG, PERS, MISC

{'B-ORG', 'I-PER', 'B-LOC', 'O', 'I-LOC', 'B-PER', 'I-ORG'}


### Ronec
https://huggingface.co/datasets/ronec


In [8]:
ronec_label_map = {
    'O': 0,
    'B-PERSON': 1,
    'I-PERSON': 2,
    'B-GPE': 3,
    'I-GPE': 4,
    'B-LOC': 5,
    'I-LOC': 6,
    'B-ORG': 7,
    'I-ORG': 8,
    'B-LANGUAGE': 9,
    'I-LANGUAGE': 10,
    'B-NAT_REL_POL': 11 ,
    'I-NAT_REL_POL': 12,
    'B-DATETIME': 13,
    'I-DATETIME': 14,
    'B-PERIOD': 15,
    'I-PERIOD': 16,
    'B-QUANTITY': 17,
    'I-QUANTITY': 18,
    'B-MONEY': 19,
    'I-MONEY': 20,
    'B-NUMERIC': 21,
    'I-NUMERIC': 22,
    'B-ORDINAL': 23,
    'I-ORDINAL': 24,
    'B-FACILITY': 25,
    'I-FACILITY': 26,
    'B-WORK_OF_ART': 27 ,
    'I-WORK_OF_ART': 28,
    'B-EVENT': 29,
    'I-EVENT': 30,
}

ronec = ner.ReadNERData()
ronec_words, ronec_labels = ronec.read_dataset('ronec', ronec_label_map)

Generating train split: 0 examples [00:00, ? examples/s]

Generating validation split: 0 examples [00:00, ? examples/s]

Generating test split: 0 examples [00:00, ? examples/s]

Generating test Split


  0%|          | 0/2000 [00:00<?, ?it/s]

In [9]:
print(ner.check_labels(ronec_labels))
label_alignment = {
    'O': 'O',
    'B-PERSON': 'B-PER',
    'I-PERSON': 'I-PER',
    'B-GPE': 'O',
    'I-GPE': 'O',
    'B-LOC': 'B-LOC',
    'I-LOC': 'I-LOC',
    'B-ORG': 'B-ORG',
    'I-ORG': 'I-ORG',
    'B-LANGUAGE': 'O',
    'I-LANGUAGE': 'O',
    'B-NAT_REL_POL': 'O',
    'I-NAT_REL_POL': 'O',
    'B-DATETIME': 'O',
    'I-DATETIME': 'O',
    'B-PERIOD': 'O',
    'I-PERIOD': 'O',
    'B-QUANTITY': 'O',
    'I-QUANTITY': 'O',
    'B-MONEY': 'O',
    'I-MONEY': 'O',
    'B-NUMERIC': 'O',
    'I-NUMERIC': 'O',
    'B-ORDINAL': 'O',
    'I-ORDINAL': 'O',
    'B-FACILITY': 'O',
    'I-FACILITY': 'O',
    'B-WORK_OF_ART': 'O',
    'I-WORK_OF_ART': 'O',
    'B-EVENT': 'O',
    'I-EVENT': 'O',
}
# Align the dataset labels to the standard labels
ronec_labels = ner.align_dataset(ronec_labels, label_alignment)
print(ner.check_labels(ronec_labels))

{'B-WORK_OF_ART', 'B-NUMERIC', 'I-LANGUAGE', 'I-LOC', 'B-EVENT', 'I-WORK_OF_ART', 'I-QUANTITY', 'I-GPE', 'B-FACILITY', 'O', 'I-EVENT', 'I-PERSON', 'B-NAT_REL_POL', 'B-PERIOD', 'B-PERSON', 'I-ORDINAL', 'B-MONEY', 'B-ORDINAL', 'B-DATETIME', 'I-MONEY', 'I-NUMERIC', 'I-DATETIME', 'I-ORG', 'B-LANGUAGE', 'I-PERIOD', 'B-ORG', 'B-LOC', 'B-GPE', 'I-NAT_REL_POL', 'B-QUANTITY', 'I-FACILITY'}
{'B-ORG', 'I-PER', 'B-LOC', 'O', 'I-LOC', 'B-PER', 'I-ORG'}


# Evaluate model

In [12]:
alignment = {
'B-DATETIME': 'O',
'B-EVENT': 'O',
'B-FACILITY': 'O',
'B-GPE': 'O',
'B-LANGUAGE': 'O',
'B-LOC': 'B-LOC',
'B-MONEY': 'O',
'B-NAT_REL_POL': 'O',
'B-NUMERIC': 'O',
'B-ORDINAL': 'O',
'B-ORG': 'B-ORG',
'B-PER': 'B-PER',
'B-PERIOD': 'O',
'B-QUANTITY': 'O',
'B-WORK_OF_ART': 'O',
'I-DATETIME': 'O',
'I-EVENT': 'O',
'I-FACILITY': 'O',
'I-GPE': 'O',
'I-LANGUAGE': 'O',
'I-LOC': 'I-LOC',
'I-MONEY': 'O',
'I-NAT_REL_POL': 'O',
'I-NUMERIC': 'O',
'I-ORDINAL': 'O',
'I-ORG': 'I-ORG',
'I-PER': 'I-PER',
'I-PERIOD': 'O',
'I-QUANTITY': 'O',
'I-WORK_OF_ART': 'O',
'O': 'O',
 }

model_name = "EvanD/xlm-roberta-base-romanian-ner-ronec"
model_name_output = 'EvanD/xlm-roberta-base-romanian-ner-ronec'
model_evaluation = ner.ModelEvaluation(
    model_name,
    alignment
)

In [11]:
model_evaluation.model.config.id2label

{0: 'B-DATETIME',
 1: 'B-EVENT',
 2: 'B-FACILITY',
 3: 'B-GPE',
 4: 'B-LANGUAGE',
 5: 'B-LOC',
 6: 'B-MONEY',
 7: 'B-NAT_REL_POL',
 8: 'B-NUMERIC',
 9: 'B-ORDINAL',
 10: 'B-ORG',
 11: 'B-PER',
 12: 'B-PERIOD',
 13: 'B-QUANTITY',
 14: 'B-WORK_OF_ART',
 15: 'I-DATETIME',
 16: 'I-EVENT',
 17: 'I-FACILITY',
 18: 'I-GPE',
 19: 'I-LANGUAGE',
 20: 'I-LOC',
 21: 'I-MONEY',
 22: 'I-NAT_REL_POL',
 23: 'I-NUMERIC',
 24: 'I-ORDINAL',
 25: 'I-ORG',
 26: 'I-PER',
 27: 'I-PERIOD',
 28: 'I-QUANTITY',
 29: 'I-WORK_OF_ART',
 30: 'O'}

### wikiann

In [13]:
data_name = "wikiann"
wikiann_evaluation_output = model_evaluation.evaluate_model(wikiann_words, wikiann_labels)

  0%|          | 0/625 [00:00<?, ?it/s]

In [14]:
wikiann_seqeval = wikiann_evaluation_output.get_classification('Seqeval')
wikiann_seqeval

,Tag,Precision,Recall,F1,support
0,LOC,0.2079,0.0250,0.0446,3804
1,ORG,0.5082,0.2971,0.3749,3666
2,PER,0.5057,0.6991,0.5869,4220
3,micro,0.4902,0.3536,0.4109,11690
4,macro,0.4073,0.3404,0.3355,11690
5,weighted,0.4096,0.3536,0.3440,11690


In [15]:
wikiann_sklearn = wikiann_evaluation_output.get_classification('Sklearn')
wikiann_sklearn

,Tag,Precision,Recall,F1,support
0,B-LOC,0.3898,0.0302,0.0561,3804
1,B-ORG,0.8193,0.2894,0.4277,3666
2,B-PER,0.5856,0.5026,0.5409,4220
3,I-LOC,0.5536,0.0409,0.0761,7583
4,I-ORG,0.7203,0.3347,0.4571,10104
5,I-PER,0.6616,0.7285,0.6935,8314
6,O,0.5760,0.9350,0.7128,28991
7,accuracy,0.6021,66682,None,None
8,macro,0.6152,0.4088,0.4235,66682
9,weighted,0.6094,0.6021,0.5352,66682


### Ronec

In [16]:
data_name = "ronec"
ronec_evaluation_output = model_evaluation.evaluate_model(ronec_words, ronec_labels)

  0%|          | 0/125 [00:00<?, ?it/s]

In [17]:
ronec_seqeval = ronec_evaluation_output.get_classification('Seqeval')
ronec_seqeval

,Tag,Precision,Recall,F1,support
0,LOC,0.0802,0.0185,0.0301,1728
1,ORG,0.0007,0.0027,0.0011,373
2,PER,0.8776,0.9033,0.8903,4230
3,micro,0.6262,0.6088,0.6173,6331
4,macro,0.3195,0.3082,0.3072,6331
5,weighted,0.6083,0.6088,0.6031,6331


In [18]:
ronec_sklearn = ronec_evaluation_output.get_classification('Sklearn')
ronec_sklearn

,Tag,Precision,Recall,F1,support
0,B-LOC,0.0949,0.0203,0.0334,1728
1,B-ORG,0.0008,0.0027,0.0012,373
2,B-PER,0.9215,0.8719,0.8960,4230
3,I-LOC,0.0299,0.0513,0.0377,273
4,I-ORG,0.0013,0.0060,0.0021,503
5,I-PER,0.8081,0.9071,0.8547,2186
6,O,0.9694,0.9498,0.9595,79176
7,accuracy,0.9148,88469,None,None
8,macro,0.4037,0.4013,0.3978,88469
9,weighted,0.9336,0.9148,0.9235,88469
